**Import Required Libraries**

In [2]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=c1a38119a45a9047002538c8517db83cec525651737ec4dec8a912e00038f3fb
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [3]:
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

from seqeval.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "parquet",
    data_files={
        "train": "https://huggingface.co/datasets/tomaarsen/conll2003/resolve/main/data/train-00000-of-00001.parquet",
        "validation": "https://huggingface.co/datasets/tomaarsen/conll2003/resolve/main/data/validation-00000-of-00001.parquet",
        "test": "https://huggingface.co/datasets/tomaarsen/conll2003/resolve/main/data/test-00000-of-00001.parquet",
    }
)

dataset

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.24MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  316kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  288kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

**Load the DistilBERT Tokenizer**

In [4]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 43
NUM_LABELS = 9

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

**Label Alignment & Tokenization**

In [5]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

**applying Label Alighnment on (train , valid, test )**

In [8]:
tokenized_train = dataset["train"].map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_validation = dataset["validation"].map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_test = dataset["test"].map(
    tokenize_and_align_labels,
    batched=True
)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [13]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [14]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Training Arguments**

In [15]:
training_args = TrainingArguments(

    output_dir="./distilbert_output",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    report_to="none"
)

**Train & Fit**

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.079435,0.067914,0.980679,0.907657,0.920159,0.913865
2,0.050434,0.063338,0.982756,0.924907,0.930197,0.927545
3,0.024586,0.061929,0.984500,0.935589,0.939302,0.937442
4,0.018374,0.066378,0.984251,0.928843,0.940119,0.934447
5,0.013497,0.067805,0.984085,0.929197,0.940586,0.934857


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4390, training_loss=0.058403068332845906, metrics={'train_runtime': 457.602, 'train_samples_per_second': 153.419, 'train_steps_per_second': 9.593, 'total_flos': 770444255931210.0, 'train_loss': 0.058403068332845906, 'epoch': 5.0})

**Evaluation**

In [17]:
def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [
            label_names[p]
            for (p, l) in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [
            label_names[l]
            for (p, l) in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "accuracy": accuracy_score(true_labels, true_predictions),
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }

In [19]:
results = trainer.evaluate(tokenized_test)

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.013497,0.125394,5,0.974168,0.888068,0.888177,0.888123


{'eval_loss': 0.12539434432983398, 'eval_accuracy': 0.9741676336695528, 'eval_precision': 0.8880679719246398, 'eval_recall': 0.8881773399014778, 'eval_f1': 0.8881226525460254}


In [20]:
import pandas as pd

results = trainer.evaluate(tokenized_test)

metrics_table = pd.DataFrame({
    "Metric": [
        "Test Loss",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Value": [
        results["eval_loss"],
        results["eval_accuracy"],
        results["eval_precision"],
        results["eval_recall"],
        results["eval_f1"]
    ]
})

metrics_table

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.013497,0.125394,5,0.974168,0.888068,0.888177,0.888123


,Metric,Value
0,Test Loss,0.125394
1,Accuracy,0.974168
2,Precision,0.888068
3,Recall,0.888177
4,F1 Score,0.888123


**Save Model**

In [25]:
trainer.save_model("Best_DistilBERT_Model")

tokenizer.save_pretrained("Best_DistilBERT_Model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('Best_DistilBERT_Model/tokenizer_config.json',
 'Best_DistilBERT_Model/tokenizer.json')

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
label_names = dataset["train"].features["ner_tags"].feature.names

print(label_names)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [27]:
label_names = list(label_names)  # لو كانت Tuple أو Sequence
label_names.append("<pad>")

print(label_names)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC', '<pad>']


In [28]:
idx2tag = {i: tag for i, tag in enumerate(label_names)}

print(idx2tag)

{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC', 9: '<pad>'}


In [29]:
import pickle

with open("idx2tag.pkl", "wb") as f:
    pickle.dump(idx2tag, f)